# DATA UNDERSTANDING — TÌM HIỂU DỮ LIỆU

**Đề tài:** Phân tích dữ liệu kinh doanh và ứng dụng học máy trong thương mại điện tử.

## 1. Giới thiệu

### 1.1. Nguồn dữ liệu

Bộ dữ liệu được sử dụng là DATATHON 2026 — Round 1, mô phỏng hoạt động kinh doanh của một doanh nghiệp thương mại điện tử thời trang tại Việt Nam.

Dữ liệu bao gồm các bảng về sản phẩm, khách hàng, đơn hàng, thanh toán, vận chuyển, hoàn trả, khuyến mãi, truy cập website, doanh thu và tồn kho.

### 1.2. Mục tiêu

Notebook này được xây dựng nhằm:

1. Khảo sát cấu trúc và quy mô của bộ dữ liệu.
2. Xác định ý nghĩa, kiểu dữ liệu và mối quan hệ giữa các bảng.
3. Kiểm tra sơ bộ chất lượng dữ liệu, bao gồm giá trị thiếu, dữ liệu trùng lặp và tính hợp lệ của các khóa.
4. Kiểm tra tính nhất quán của dữ liệu giao dịch và tồn kho.
5. Ghi nhận các vấn đề cần xác minh hoặc xử lý ở những bước tiếp theo.

### 1.3. Phạm vi thực hiện

Notebook tập trung vào giai đoạn Data Understanding trong quy trình thực hiện đồ án.

**Thứ tự đọc trong GD1.** Đọc mục 1–5.1 của notebook này để nắm cấu trúc, chất lượng và liên kết của dữ liệu. Đọc tiếp [GD1_tong_hop_kiem_chung.ipynb](GD1_tong_hop_kiem_chung.ipynb) để xem V01–V09. Các mục 6–7 bên dưới giúp tra cứu cách nhóm đối chiếu giao dịch và tồn kho ban đầu.

Các kết quả kiểm tra được sử dụng để đánh giá mức độ phù hợp của dữ liệu và chuẩn bị cho bước Exploratory Data Analysis (EDA).

Ở giai đoạn này, dữ liệu gốc được giữ nguyên. Các vấn đề phát hiện sẽ được ghi nhận, chưa tự động xóa hoặc điều chỉnh.

### 1.4. Nguyên tắc đánh giá

- Phân biệt giá trị bất thường với dữ liệu sai.
- Không mặc định giá trị thiếu luôn cần xử lý.
- Kiểm tra mối quan hệ giữa các bảng trước khi kết hợp dữ liệu.
- Không đưa ra kết luận nghiệp vụ khi chưa có đủ bằng chứng.
- Lưu kết quả kiểm tra để phục vụ báo cáo và các bước tiếp theo.

## 2. Nạp dữ liệu (Data Loading)

### 2.1. Import thư viện và thiết lập đường dẫn

**Mục tiêu:** Chuẩn bị môi trường làm việc, xác định thư mục chứa dữ liệu và thiết lập các thư viện cần thiết cho quá trình tìm hiểu dữ liệu.

**Phương pháp thực hiện:**

- Sử dụng `pandas` để đọc, kiểm tra và thao tác với dữ liệu dạng bảng.
- Sử dụng `numpy` để hỗ trợ các phép tính và xử lý dữ liệu số.
- Sử dụng `pathlib` để quản lý đường dẫn tệp và thư mục.

Dữ liệu gốc được lưu trong thư mục `data/raw/` và không bị chỉnh sửa trong quá trình tìm hiểu dữ liệu.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

start = Path(os.environ.get("RETAIL_ANALYTICS_ROOT") or Path.cwd()).expanduser().resolve()
PROJECT_DIR = next(
    (folder for folder in (start, *start.parents) if (folder / "data" / "raw").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy data/raw/. Hãy mở thư mục gốc repo hoặc đặt RETAIL_ANALYTICS_ROOT."
    )

DATA_DIR = PROJECT_DIR / "data" / "raw"
print("Thư mục dữ liệu:", DATA_DIR.relative_to(PROJECT_DIR).as_posix())


Thư mục dữ liệu: data/raw


### 2.2 Liệt kê tất cả các tệp

In [2]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print("Số file CSV:", len(csv_files))

for file in csv_files:
    print(file.name)

Số file CSV: 14
customers.csv
geography.csv
inventory.csv
order_items.csv
orders.csv
payments.csv
products.csv
promotions.csv
returns.csv
reviews.csv
sales.csv
sample_submission.csv
shipments.csv
web_traffic.csv


### 2.3 Đọc tất cả các bảng

In [3]:
# Danh sách 13 bảng dữ liệu nghiệp vụ
business_tables = [
    "products",
    "customers",
    "promotions",
    "geography",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "returns",
    "reviews",
    "sales",
    "inventory",
    "web_traffic"
]

# Tệp mẫu nộp bài được khảo sát cùng các CSV nguồn.
source_tables = [*business_tables, "sample_submission"]
missing_files = [f"{name}.csv" for name in source_tables
                 if not (DATA_DIR / f"{name}.csv").is_file()]
if missing_files:
    raise FileNotFoundError("Thiếu CSV trong data/raw/: " + ", ".join(missing_files))

datasets = {}
for table in source_tables:
    df = pd.read_csv(DATA_DIR / f"{table}.csv", low_memory=False, encoding="utf-8-sig")
    datasets[table] = df
    print(f"{table}: {df.shape[0]:,} dòng, {df.shape[1]} cột")


products: 2,412 dòng, 8 cột
customers: 121,930 dòng, 7 cột
promotions: 50 dòng, 10 cột
geography: 39,948 dòng, 4 cột
orders: 646,945 dòng, 8 cột
order_items: 714,669 dòng, 7 cột
payments: 646,945 dòng, 4 cột
shipments: 566,067 dòng, 4 cột
returns: 39,939 dòng, 7 cột
reviews: 113,551 dòng, 7 cột
sales: 3,833 dòng, 3 cột
inventory: 60,247 dòng, 17 cột
web_traffic: 3,652 dòng, 7 cột
sample_submission: 548 dòng, 3 cột


### 2.4. Mẫu bài nộp dự báo

`sample_submission.csv` cho biết lịch và tên các cột cần nộp trong Bài toán 3. Dùng `Date` để xác định các ngày cần dự báo. Hai cột `Revenue` và `COGS` trong tệp chứa giá trị mẫu.

In [4]:
submission = datasets["sample_submission"]
submission_dates = pd.to_datetime(submission["Date"], format="%Y-%m-%d", errors="raise")
display(pd.DataFrame([{
    "Số dòng": len(submission),
    "Các cột": ", ".join(submission.columns),
    "Ngày đầu": submission_dates.min().date(),
    "Ngày cuối": submission_dates.max().date(),
    "Số ngày phân biệt": submission_dates.nunique(),
}]))


,Số dòng,Các cột,Ngày đầu,Ngày cuối,Số ngày phân biệt
0,548,"Date, Revenue, COGS",2023-01-01,2024-07-01,548


## 3. Khảo sát cấu trúc dữ liệu (Data Structure Overview)

### 3.1. Tổng quan quy mô dữ liệu

In [5]:
overview = []

for name, df in datasets.items():
    overview.append({
        "Table": name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Missing_Cells": int(df.isna().sum().sum()),
        "Duplicate_Rows": int(df.duplicated().sum()),
        "Memory_MB": round(
            df.memory_usage(deep=True).sum() / 1024**2, 2
        )
    })

overview_df = pd.DataFrame(overview)

display(overview_df)

,Table,Rows,Columns,Missing_Cells,Duplicate_Rows,Memory_MB
0,products,2412,8,0,0,0.71
1,customers,121930,7,0,0,34.86
2,promotions,50,10,40,0,0.02
3,geography,39948,4,0,0,6.86
4,orders,646945,8,0,0,194.24
5,order_items,714669,7,1152816,0,78.00
6,payments,646945,4,0,0,50.56
7,shipments,566067,4,0,0,72.34
8,returns,39939,7,0,0,8.02
9,reviews,113551,7,0,0,23.28


### 3.2 Kiểm tra tên cột và kiểu dữ liệu

In [6]:
column_info = pd.concat(
    [
        pd.DataFrame({
            "Table": name,
            "Column": df.columns,
            "Data_Type": df.dtypes.astype(str).values
        })
        for name, df in datasets.items()
    ],
    ignore_index=True
)

display(column_info)

,Table,Column,Data_Type
0,products,product_id,int64
1,products,product_name,object
2,products,category,object
3,products,segment,object
4,products,size,object
...,...,...,...
91,web_traffic,avg_session_duration_sec,float64
92,web_traffic,traffic_source,object
93,sample_submission,Date,object
94,sample_submission,Revenue,float64


Kiểm tra riêng các cột thời gian

In [7]:
# Danh sách các cột thời gian cần khảo sát
date_columns = {
    "customers": ["signup_date"],
    "promotions": ["start_date", "end_date"],
    "orders": ["order_date"],
    "shipments": ["ship_date", "delivery_date"],
    "returns": ["return_date"],
    "reviews": ["review_date"],
    "sales": ["Date"],
    "sample_submission": ["Date"],
    "inventory": ["snapshot_date"],
    "web_traffic": ["date"]
}

date_type_report = []

for table, columns in date_columns.items():
    df = datasets[table]

    for col in columns:
        if col in df.columns:
            date_type_report.append({
                "Table": table,
                "Column": col,
                "Current_Type": str(df[col].dtype),
                "Sample_Value": (
                    df[col].dropna().iloc[0]
                    if df[col].notna().any()
                    else None
                )
            })

date_type_report = pd.DataFrame(date_type_report)

display(date_type_report)

,Table,Column,Current_Type,Sample_Value
0,customers,signup_date,object,12/30/2021
1,promotions,start_date,object,2013-03-18
2,promotions,end_date,object,2013-04-17
3,orders,order_date,object,2012-07-04
4,shipments,ship_date,object,2012-07-07
5,shipments,delivery_date,object,2012-07-11
6,returns,return_date,object,2012-07-25
7,reviews,review_date,object,2012-07-24
8,sales,Date,object,2012-07-04
9,sample_submission,Date,object,2023-01-01


### 3.3 Quan sát dữ liệu mẫu (Data Preview)

In [8]:
# Xem 5 dòng đầu tiên của từng bảng
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"TABLE: {name.upper()}")
    print(f"Shape: {df.shape}")
    print(f"{'=' * 60}")

    display(df.head())


TABLE: PRODUCTS
Shape: (2412, 8)

TABLE: CUSTOMERS
Shape: (121930, 7)

TABLE: PROMOTIONS
Shape: (50, 10)

TABLE: GEOGRAPHY
Shape: (39948, 4)

TABLE: ORDERS
Shape: (646945, 8)

TABLE: ORDER_ITEMS
Shape: (714669, 7)

TABLE: PAYMENTS
Shape: (646945, 4)

TABLE: SHIPMENTS
Shape: (566067, 4)

TABLE: RETURNS
Shape: (39939, 7)

TABLE: REVIEWS
Shape: (113551, 7)

TABLE: SALES
Shape: (3833, 3)

TABLE: INVENTORY
Shape: (60247, 17)

TABLE: WEB_TRAFFIC
Shape: (3652, 7)

TABLE: SAMPLE_SUBMISSION
Shape: (548, 3)


,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717300,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,12/30/2021,Female,35-44,social_media
1,2,15201,Hai Phong,12/27/2013,Female,45-54,email_campaign
2,3,15201,Hai Phong,7/24/2018,Female,18-24,organic_search
3,4,15201,Hai Phong,11/29/2017,Male,35-44,referral
4,5,15201,Hai Phong,9/23/2022,Male,55+,organic_search


,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,NaN,email,1,0
1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,NaN,online,0,0
2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,NaN,email,0,0
3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,NaN,all_channels,0,50000
4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000


,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13
3,15204,Bac Giang,East,District #13
4,15205,Bac Giang,East,District #13


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


... (5 outputs left out)

,Date,Revenue,COGS
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79


,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,fill_rate,stockout_flag,overstock_flag,reorder_flag,sell_through_rate,product_name,category,segment,year,month
0,2022-10-31,1,3,1,1,2,90.0,0.9333,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,10
1,2022-11-30,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,11
2,2022-12-31,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,12
3,2016-04-30,3,35,13,11,2,95.5,0.9333,1,1,0,0.2391,DragonWear MA-03,Casual,All-weather,2016,4
4,2016-05-31,3,36,11,10,1,108.0,0.9667,1,1,0,0.2174,DragonWear MA-03,Casual,All-weather,2016,5


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral


,Date,Revenue,COGS
0,2023-01-01,2665507.20,2518885.15
1,2023-01-02,1280007.89,1136463.00
2,2023-01-03,1015899.51,822721.12
3,2023-01-04,1142997.27,914554.18
4,2023-01-05,1236312.34,984390.24


### 3.4. Xác định đơn vị quan sát và khóa dữ liệu

In [9]:
table_granularity = pd.DataFrame([
    ["products", "Một sản phẩm", "product_id"],
    ["customers", "Một khách hàng", "customer_id"],
    ["promotions", "Một chương trình khuyến mãi", "promo_id"],
    ["geography", "Một mã bưu chính", "zip"],
    ["orders", "Một đơn hàng", "order_id"],
    ["order_items", "Một dòng sản phẩm trong đơn hàng", "Cần xác minh khóa dòng"],
    ["payments", "Một bản ghi thanh toán của đơn hàng", "order_id"],
    ["shipments", "Một bản ghi vận chuyển của đơn hàng", "order_id"],
    ["returns", "Một bản ghi trả hàng", "return_id"],
    ["reviews", "Một bản ghi đánh giá", "review_id"],
    ["sales", "Một ngày", "Date"],
    ["sample_submission", "Một ngày trong lịch nộp bài", "Date"],
    ["inventory", "Một sản phẩm tại một tháng", "product_id + snapshot_date"],
    ["web_traffic", "Một ngày và nguồn truy cập", "date + traffic_source"]
], columns=[
    "Table",
    "Granularity",
    "Expected_Key"
])

display(table_granularity)

,Table,Granularity,Expected_Key
0,products,Một sản phẩm,product_id
1,customers,Một khách hàng,customer_id
2,promotions,Một chương trình khuyến mãi,promo_id
3,geography,Một mã bưu chính,zip
4,orders,Một đơn hàng,order_id
5,order_items,Một dòng sản phẩm trong đơn hàng,Cần xác minh khóa dòng
6,payments,Một bản ghi thanh toán của đơn hàng,order_id
7,shipments,Một bản ghi vận chuyển của đơn hàng,order_id
8,returns,Một bản ghi trả hàng,return_id
9,reviews,Một bản ghi đánh giá,review_id


## 4. Kiểm tra chất lượng dữ liệu tổng quan (Data Quality Check)

### 4.1 Kiểm tra giá trị thiếu (Missing Values)

In [10]:
missing_records = []

for table_name, df in datasets.items():
    for col in df.columns:
        missing_count = int(df[col].isna().sum())
        missing_pct = df[col].isna().mean() * 100

        missing_records.append({
            "Table": table_name,
            "Column": col,
            "Missing_Count": missing_count,
            "Missing_Percent": round(missing_pct, 2)
        })

missing_report = pd.DataFrame(missing_records)

# Chỉ hiển thị những cột thực sự có giá trị thiếu
missing_columns = (
    missing_report[missing_report["Missing_Count"] > 0]
    .sort_values(
        ["Missing_Percent", "Missing_Count"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("Số cột có giá trị thiếu:", len(missing_columns))

display(missing_columns)

Số cột có giá trị thiếu: 3


,Table,Column,Missing_Count,Missing_Percent
0,order_items,promo_id_2,714463,99.97
1,promotions,applicable_category,40,80.00
2,order_items,promo_id,438353,61.34


### 4.2 Kiểm tra dòng trùng lặp (Duplicate Records)

In [11]:
duplicate_report = pd.DataFrame([
    {
        "Table": table_name,
        "Total_Rows": len(df),
        "Duplicate_Rows": int(df.duplicated().sum()),
        "Duplicate_Percent": round(
            df.duplicated().mean() * 100, 2
        )
    }
    for table_name, df in datasets.items()
])

duplicate_report = duplicate_report.sort_values(
    "Duplicate_Rows",
    ascending=False
).reset_index(drop=True)

display(duplicate_report)

,Table,Total_Rows,Duplicate_Rows,Duplicate_Percent
0,products,2412,0,0.0
1,customers,121930,0,0.0
2,promotions,50,0,0.0
3,geography,39948,0,0.0
4,orders,646945,0,0.0
5,order_items,714669,0,0.0
6,payments,646945,0,0.0
7,shipments,566067,0,0.0
8,returns,39939,0,0.0
9,reviews,113551,0,0.0


### 4.3 Kiểm tra khóa chính (Primary Key Validation)

Bảng dưới kiểm tra các khóa dự kiến trên bộ dữ liệu đang có. Cặp `order_id` và `product_id` trong `order_items` được kiểm tra riêng vì một đơn có thể chứa nhiều dòng cùng sản phẩm.


In [12]:
# Khóa chính / khóa nghiệp vụ dự kiến
primary_keys = {
    "products": ["product_id"],
    "customers": ["customer_id"],
    "promotions": ["promo_id"],
    "geography": ["zip"],
    "orders": ["order_id"],
    "payments": ["order_id"],
    "shipments": ["order_id"],
    "returns": ["return_id"],
    "reviews": ["review_id"],
    "sales": ["Date"],
    "sample_submission": ["Date"],
    "inventory": ["product_id", "snapshot_date"],
    "web_traffic": ["date", "traffic_source"]
}

pk_results = []

for table, keys in primary_keys.items():
    df = datasets[table]

    missing_key_rows = df[keys].isna().any(axis=1).sum()
    duplicate_key_rows = df.duplicated(
        subset=keys,
        keep=False
    ).sum()

    pk_results.append({
        "Table": table,
        "Key": " + ".join(keys),
        "Missing_Key_Rows": int(missing_key_rows),
        "Rows_With_Duplicate_Key": int(duplicate_key_rows),
        "Status": (
            "PASS"
            if missing_key_rows == 0 and duplicate_key_rows == 0
            else "CHECK"
        )
    })

pk_report = pd.DataFrame(pk_results)

display(pk_report)

item_pair = ["order_id", "product_id"]
item_rows = datasets["order_items"]
repeated_pairs = item_rows.loc[item_rows.duplicated(subset=item_pair, keep=False), item_pair]
display(pd.DataFrame([{
    "Bảng": "order_items",
    "Dòng trùng toàn bộ": int(item_rows.duplicated().sum()),
    "Số cặp đơn–sản phẩm lặp": len(repeated_pairs.drop_duplicates()),
    "Số dòng thuộc các cặp lặp": len(repeated_pairs),
}]))


,Table,Key,Missing_Key_Rows,Rows_With_Duplicate_Key,Status
0,products,product_id,0,0,PASS
1,customers,customer_id,0,0,PASS
2,promotions,promo_id,0,0,PASS
3,geography,zip,0,0,PASS
4,orders,order_id,0,0,PASS
5,payments,order_id,0,0,PASS
6,shipments,order_id,0,0,PASS
7,returns,return_id,0,0,PASS
8,reviews,review_id,0,0,PASS
9,sales,Date,0,0,PASS


,Bảng,Dòng trùng toàn bộ,Số cặp đơn–sản phẩm lặp,Số dòng thuộc các cặp lặp
0,order_items,0,16,32


### 4.4 Kiểm tra khóa ngoại (Foreign Key Validation)

In [13]:
# (Bảng nguồn, cột FK, bảng tham chiếu, cột PK)
relationships = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "zip", "geography", "zip"),
    ("customers", "zip", "geography", "zip"),

    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),

    ("payments", "order_id", "orders", "order_id"),
    ("shipments", "order_id", "orders", "order_id"),

    ("returns", "order_id", "orders", "order_id"),
    ("returns", "product_id", "products", "product_id"),

    ("reviews", "order_id", "orders", "order_id"),
    ("reviews", "product_id", "products", "product_id"),
    ("reviews", "customer_id", "customers", "customer_id"),

    ("inventory", "product_id", "products", "product_id"),

    ("order_items", "promo_id", "promotions", "promo_id"),
    ("order_items", "promo_id_2", "promotions", "promo_id")
]

fk_results = []

for source_table, fk_col, target_table, pk_col in relationships:
    source = datasets[source_table]
    target = datasets[target_table]

    fk_values = source[fk_col]
    valid_values = target[pk_col].dropna()

    missing_mask = fk_values.isna()

    invalid_mask = (
        fk_values.notna()
        & ~fk_values.isin(valid_values)
    )

    fk_results.append({
        "Source_Table": source_table,
        "Foreign_Key": fk_col,
        "Target_Table": target_table,
        "Reference_Key": pk_col,
        "Missing_FK": int(missing_mask.sum()),
        "Invalid_FK": int(invalid_mask.sum()),
        "Status": (
            "PASS" if invalid_mask.sum() == 0 else "CHECK"
        )
    })

fk_report = pd.DataFrame(fk_results)

display(fk_report)

,Source_Table,Foreign_Key,Target_Table,Reference_Key,Missing_FK,Invalid_FK,Status
0,orders,customer_id,customers,customer_id,0,0,PASS
1,orders,zip,geography,zip,0,0,PASS
2,customers,zip,geography,zip,0,0,PASS
3,order_items,order_id,orders,order_id,0,0,PASS
4,order_items,product_id,products,product_id,0,0,PASS
5,payments,order_id,orders,order_id,0,0,PASS
6,shipments,order_id,orders,order_id,0,0,PASS
7,returns,order_id,orders,order_id,0,0,PASS
8,returns,product_id,products,product_id,0,0,PASS
9,reviews,order_id,orders,order_id,0,0,PASS


### 5.1 Khảo sát phạm vi thời gian (Temporal Coverage)

In [14]:
temporal_results = []

for table_name, columns in date_columns.items():
    df = datasets[table_name]

    for col in columns:
        if col not in df.columns:
            print(f"Không tìm thấy: {table_name}.{col}")
            continue

        # Đọc định dạng của CSV nguồn trên Series tạm thời.
        date_format = "%m/%d/%Y" if col == "signup_date" else "%Y-%m-%d"
        dates = pd.to_datetime(df[col], format=date_format, errors="coerce")

        temporal_results.append({
            "Table": table_name,
            "Date_Column": col,
            "Start_Date": dates.min(),
            "End_Date": dates.max(),
            "Unique_Days": dates.dt.normalize().nunique(),
            "Unique_Months": dates.dt.to_period("M").nunique(),
            "Invalid_Or_Missing": int(dates.isna().sum())
        })

temporal_report = pd.DataFrame(temporal_results)

display(temporal_report)

,Table,Date_Column,Start_Date,End_Date,Unique_Days,Unique_Months,Invalid_Or_Missing
0,customers,signup_date,2012-01-17,2022-12-31,3941,132,0
1,promotions,start_date,2013-01-31,2022-11-18,50,50,0
2,promotions,end_date,2013-03-01,2022-12-31,50,50,0
3,orders,order_date,2012-07-04,2022-12-31,3833,126,0
4,shipments,ship_date,2012-07-04,2022-12-29,3831,126,0
5,shipments,delivery_date,2012-07-06,2022-12-31,3831,126,0
6,returns,return_date,2012-07-11,2022-12-31,3806,126,0
7,reviews,review_date,2012-07-10,2022-12-31,3825,126,0
8,sales,Date,2012-07-04,2022-12-31,3833,126,0
9,sample_submission,Date,2023-01-01,2024-07-01,548,19,0


### 5.2 Khảo sát trạng thái đơn hàng (Order Status Overview)

In [15]:
orders = datasets["orders"]

order_status_report = (
    orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("Order_Status")
    .reset_index(name="Order_Count")
)

order_status_report["Percentage"] = (
    order_status_report["Order_Count"]
    / len(orders)
    * 100
).round(2)

display(order_status_report)

,Order_Status,Order_Count,Percentage
0,delivered,516716,79.87
1,cancelled,59462,9.19
2,returned,36142,5.59
3,shipped,13773,2.13
4,paid,13577,2.10
5,created,7275,1.12


### 5.3 Khảo sát khách hàng có giao dịch (Customer Transaction Coverage)

In [16]:
customers = datasets["customers"]
orders = datasets["orders"]

# Danh sách khách hàng trong bảng danh mục
customer_ids = set(
    customers["customer_id"].dropna().unique()
)

# Danh sách khách hàng xuất hiện trong đơn hàng
ordering_customer_ids = set(
    orders["customer_id"].dropna().unique()
)

# Khách hàng có đơn hàng và có trong danh mục
customers_with_orders = customer_ids & ordering_customer_ids

# Khách hàng có trong danh mục nhưng chưa có đơn hàng
customers_without_orders = customer_ids - ordering_customer_ids

# Mã khách hàng có trong đơn hàng nhưng không có trong danh mục
unknown_customer_ids = ordering_customer_ids - customer_ids

customer_coverage_report = pd.DataFrame({
    "Metric": [
        "Total_Customers",
        "Customers_With_Orders",
        "Customers_Without_Orders",
        "Unknown_Customer_IDs"
    ],
    "Count": [
        len(customer_ids),
        len(customers_with_orders),
        len(customers_without_orders),
        len(unknown_customer_ids)
    ]
})

display(customer_coverage_report)

print(
    "Tỷ lệ khách hàng có ít nhất một đơn hàng:",
    round(
        len(customers_with_orders) / len(customer_ids) * 100,
        2
    ),
    "%"
)

Tỷ lệ khách hàng có ít nhất một đơn hàng: 74.01 %


,Metric,Count
0,Total_Customers,121930
1,Customers_With_Orders,90246
2,Customers_Without_Orders,31684
3,Unknown_Customer_IDs,0


### 5.4 Khảo sát mức độ bao phủ sản phẩm (Product Coverage)

In [17]:
products = datasets["products"]
order_items = datasets["order_items"]
inventory = datasets["inventory"]

product_ids = set(
    products["product_id"].dropna().unique()
)

ordered_product_ids = set(
    order_items["product_id"].dropna().unique()
)

inventory_product_ids = set(
    inventory["product_id"].dropna().unique()
)

product_coverage_report = pd.DataFrame({
    "Metric": [
        "Products_In_Catalog",
        "Products_In_Order_Items",
        "Products_In_Inventory",
        "Catalog_Products_Without_Order_Items",
        "Catalog_Products_Without_Inventory",
        "Ordered_Products_Without_Inventory",
        "Inventory_Products_Without_Order_Items"
    ],
    "Count": [
        len(product_ids),
        len(ordered_product_ids),
        len(inventory_product_ids),
        len(product_ids - ordered_product_ids),
        len(product_ids - inventory_product_ids),
        len(ordered_product_ids - inventory_product_ids),
        len(inventory_product_ids - ordered_product_ids)
    ]
})

display(product_coverage_report)

,Metric,Count
0,Products_In_Catalog,2412
1,Products_In_Order_Items,1598
2,Products_In_Inventory,1624
3,Catalog_Products_Without_Order_Items,814
4,Catalog_Products_Without_Inventory,788
5,Ordered_Products_Without_Inventory,0
6,Inventory_Products_Without_Order_Items,26


## 6. Kiểm tra tính nhất quán dữ liệu giao dịch (Transaction Consistency Check)

### 6.1. Kiểm tra số lượng và giá trị dòng sản phẩm

**Mục tiêu:** Phát hiện các giá trị số lượng, đơn giá và giảm giá cần được xem xét trong bảng `order_items`.

**Ý nghĩa nghiệp vụ:** Những giá trị bất thường có thể ảnh hưởng đến việc tính toán số lượng bán, giá trị đơn hàng và doanh thu.

**Phương pháp:** Kiểm tra các điều kiện `quantity <= 0`, `unit_price < 0`, `discount_amount < 0` và `discount_amount > quantity × unit_price`.

**Giới hạn:** Các điều kiện trên được sử dụng như quy tắc kiểm tra sơ bộ. Việc đánh giá một bản ghi là lỗi cần căn cứ vào quy tắc nghiệp vụ và cách dữ liệu được tạo.

In [18]:
order_items = datasets["order_items"]

# Giá trị hàng hóa trước giảm giá
gross_line_value = (
    order_items["quantity"] * order_items["unit_price"]
)

transaction_value_report = pd.DataFrame({
    "Check": [
        "Quantity <= 0",
        "Unit_Price < 0",
        "Discount_Amount < 0",
        "Discount > Gross_Line_Value"
    ],
    "Count": [
        int((order_items["quantity"] <= 0).sum()),
        int((order_items["unit_price"] < 0).sum()),
        int((order_items["discount_amount"] < 0).sum()),
        int((
            order_items["discount_amount"] > gross_line_value
        ).sum())
    ]
})

display(transaction_value_report)

,Check,Count
0,Quantity <= 0,0
1,Unit_Price < 0,0
2,Discount_Amount < 0,0
3,Discount > Gross_Line_Value,0


### 6.2. Kiểm tra số dòng sản phẩm trong đơn hàng

In [19]:
items_per_order = (
    order_items
    .groupby("order_id")
    .size()
    .rename("Item_Lines")
)

print("Thống kê số dòng sản phẩm trong mỗi đơn:")
display(items_per_order.describe())

print("\nPhân bố số dòng sản phẩm:")
display(
    items_per_order
    .value_counts()
    .sort_index()
    .rename_axis("Item_Lines")
    .reset_index(name="Order_Count")
)

print("\nSố đơn có nhiều hơn một dòng sản phẩm:", int((items_per_order > 1).sum()))

Thống kê số dòng sản phẩm trong mỗi đơn:

Phân bố số dòng sản phẩm:

Số đơn có nhiều hơn một dòng sản phẩm: 66457


count    646945.000000
mean          1.104683
std           0.312812
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           5.000000
Name: Item_Lines, dtype: float64

,Item_Lines,Order_Count
0,1,580488
1,2,65256
2,3,1137
3,4,62
4,5,2


Đơn hàng không có dòng sản phẩm

In [20]:
orders = datasets["orders"]

orders_without_items = (
    ~orders["order_id"].isin(order_items["order_id"])
)

print("Số đơn hàng không có dòng sản phẩm:", int(orders_without_items.sum()))

Số đơn hàng không có dòng sản phẩm: 0


### 6.3. Đối chiếu giá trị đơn hàng và thanh toán

Giá trị từng dòng sản phẩm bằng `quantity × unit_price − discount_amount`, tổng hợp theo `order_id` và so sánh với `payment_value`.

Tính giá trị đơn hàng

In [21]:
payments = datasets["payments"]

# Tạo bản sao để không thay đổi DataFrame gốc
order_items_check = order_items.copy()

# Giá trị từng dòng sau giảm giá
order_items_check["line_value"] = (
    order_items_check["quantity"]
    * order_items_check["unit_price"]
    - order_items_check["discount_amount"]
)

# Tổng giá trị theo đơn hàng
order_values = (
    order_items_check
    .groupby("order_id", as_index=False)
    .agg(
        calculated_order_value=("line_value", "sum")
    )
)

display(order_values.head())

,order_id,calculated_order_value
0,1,7967.54
1,2,71163.75
2,3,33660.99
3,4,53196.25
4,6,1597.84


Đối chiếu với Payments

In [22]:
# Kiểm tra cấp độ order_id của payments
print("Số order_id trùng trong payments:", int(payments.duplicated("order_id").sum()))

# Dùng outer join để không bỏ sót đơn chỉ xuất hiện ở một bảng
payment_reconciliation = order_values.merge(
    payments[["order_id", "payment_value"]],
    on="order_id",
    how="outer",
    validate="one_to_one",
    indicator=True
)

# Tính sai lệch
payment_reconciliation["difference"] = (
    payment_reconciliation["calculated_order_value"]
    - payment_reconciliation["payment_value"]
)

# Chỉ so sánh số tiền khi có dữ liệu ở cả hai bảng
matched = payment_reconciliation["_merge"].eq("both")

tolerance = 0.01

amount_mismatch = (
    matched
    & payment_reconciliation["difference"].abs().gt(tolerance)
)

payment_report = pd.DataFrame({
    "Check": [
        "Orders_Only_In_Order_Items",
        "Orders_Only_In_Payments",
        "Orders_In_Both_Tables",
        "Amount_Mismatches",
        "Maximum_Absolute_Difference"
    ],
    "Result": [
        int((payment_reconciliation["_merge"] == "left_only").sum()),
        int((payment_reconciliation["_merge"] == "right_only").sum()),
        int(matched.sum()),
        int(amount_mismatch.sum()),
        payment_reconciliation.loc[
            matched, "difference"
        ].abs().max()
    ]
})

display(payment_report)

Số order_id trùng trong payments: 0


,Check,Result
0,Orders_Only_In_Order_Items,0.000000e+00
1,Orders_Only_In_Payments,0.000000e+00
2,Orders_In_Both_Tables,6.469450e+05
3,Amount_Mismatches,0.000000e+00
4,Maximum_Absolute_Difference,5.820766e-11


### 6.4 Kiểm tra dữ liệu hoàn trả (Returns Validation)

Kiểm tra số lượng và tiền hoàn

In [23]:
returns = datasets["returns"]

return_basic_report = pd.DataFrame({
    "Check": [
        "Return_Quantity <= 0",
        "Refund_Amount < 0",
        "Missing_Return_Date"
    ],
    "Count": [
        int((returns["return_quantity"] <= 0).sum()),
        int((returns["refund_amount"] < 0).sum()),
        int(returns["return_date"].isna().sum())
    ]
})

display(return_basic_report)

,Check,Count
0,Return_Quantity <= 0,0
1,Refund_Amount < 0,0
2,Missing_Return_Date,0


Đối chiếu số lượng trả với số lượng mua

In [24]:
# Tổng số lượng đặt mua theo đơn hàng và sản phẩm
ordered_quantity = (
    order_items
    .groupby(["order_id", "product_id"], as_index=False)
    .agg(ordered_qty=("quantity", "sum"))
)

# Tổng số lượng trả theo đơn hàng và sản phẩm
returned_quantity = (
    returns
    .groupby(["order_id", "product_id"], as_index=False)
    .agg(returned_qty=("return_quantity", "sum"))
)

# Giữ lại toàn bộ tổ hợp có phát sinh trả hàng
return_quantity_check = returned_quantity.merge(
    ordered_quantity,
    on=["order_id", "product_id"],
    how="left",
    validate="one_to_one",
    indicator=True
)

missing_original_item = (
    return_quantity_check["_merge"] == "left_only"
)

excessive_returns = (
    return_quantity_check["ordered_qty"].notna()
    & (
        return_quantity_check["returned_qty"]
        > return_quantity_check["ordered_qty"]
    )
)

return_quantity_report = pd.DataFrame({
    "Check": [
        "Returned_Product_Not_In_Original_Order",
        "Returned_Quantity_Exceeds_Ordered"
    ],
    "Count": [
        int(missing_original_item.sum()),
        int(excessive_returns.sum())
    ]
})

display(return_quantity_report)

,Check,Count
0,Returned_Product_Not_In_Original_Order,0
1,Returned_Quantity_Exceeds_Ordered,0


Kiểm tra ngày hoàn trả có trước ngày đặt hàng không

In [25]:
# Lấy thông tin ngày đặt hàng và ngày hoàn trả
return_date_check = returns[
    ["return_id", "order_id", "return_date"]
].merge(
    orders[["order_id", "order_date"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

# Chuyển kiểu dữ liệu ngày tháng
return_date_check["return_date"] = pd.to_datetime(
    return_date_check["return_date"], errors="coerce"
)

return_date_check["order_date"] = pd.to_datetime(
    return_date_check["order_date"], errors="coerce"
)

# Kiểm tra ngày hoàn trả trước ngày đặt hàng
invalid_return_dates = return_date_check[
    return_date_check["return_date"]
    < return_date_check["order_date"]
]

print("Số bản ghi có ngày hoàn trả trước ngày đặt hàng:", len(invalid_return_dates))

Số bản ghi có ngày hoàn trả trước ngày đặt hàng: 0


### 6.5 Đối chiếu trạng thái đơn hàng và dữ liệu hoàn trả

Đối chiếu hai bảng

In [26]:
# Tập mã đơn hàng có trạng thái returned
returned_status_ids = set(
    orders.loc[
        orders["order_status"] == "returned",
        "order_id"
    ].dropna()
)

# Tập mã đơn hàng có bản ghi trong returns
return_record_ids = set(
    returns["order_id"].dropna()
)

# Có trạng thái returned nhưng không có bản ghi hoàn trả
status_without_return = (
    returned_status_ids - return_record_ids
)

# Có bản ghi hoàn trả nhưng trạng thái không phải returned
return_without_status = (
    return_record_ids - returned_status_ids
)

return_status_report = pd.DataFrame({
    "Check": [
        "Orders_With_Returned_Status",
        "Orders_With_Return_Records",
        "Returned_Status_Without_Return_Record",
        "Return_Record_Without_Returned_Status"
    ],
    "Count": [
        len(returned_status_ids),
        len(return_record_ids),
        len(status_without_return),
        len(return_without_status)
    ]
})

display(return_status_report)

,Check,Count
0,Orders_With_Returned_Status,36142
1,Orders_With_Return_Records,36062
2,Returned_Status_Without_Return_Record,80
3,Return_Record_Without_Returned_Status,0


Xem các đơn cần kiểm tra

In [27]:
# Các đơn có trạng thái returned nhưng không có bản ghi returns
orders_missing_return_records = orders[
    orders["order_id"].isin(status_without_return)
].copy()

print(
    "Số đơn returned không có bản ghi hoàn trả:",
    len(orders_missing_return_records)
)

display(orders_missing_return_records.head(20))

Số đơn returned không có bản ghi hoàn trả: 80


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
644769,831459,2022-12-06,29847,48105,returned,credit_card,tablet,social_media
644813,831517,2022-12-06,50641,27565,returned,credit_card,desktop,organic_search
644856,831579,2022-12-09,132962,93907,returned,credit_card,mobile,paid_search
644996,831770,2022-12-12,47563,31015,returned,credit_card,tablet,direct
645007,831787,2022-12-11,73543,37329,returned,apple_pay,desktop,organic_search
645023,831807,2022-12-09,121737,62002,returned,apple_pay,desktop,email_campaign
645125,831941,2022-12-11,129887,78332,returned,cod,desktop,organic_search
645132,831948,2022-12-10,107496,80439,returned,cod,mobile,social_media
645189,832014,2022-12-15,28632,44452,returned,cod,desktop,direct
645190,832015,2022-12-11,78309,47620,returned,credit_card,desktop,email_campaign


### 6.6 Kiểm tra tính nhất quán giữa đơn hàng, khách hàng và vận chuyển

In [28]:
# Kiểm tra ngày đặt hàng và ngày đăng ký khách hàng
orders_check = datasets["orders"].merge(
    datasets["customers"][["customer_id", "signup_date"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

order_dates = pd.to_datetime(orders_check["order_date"], format="%Y-%m-%d")
signup_dates = pd.to_datetime(orders_check["signup_date"], format="%m/%d/%Y")

orders_before_signup = (order_dates < signup_dates).sum()

# Kiểm tra đơn delivered thiếu thông tin vận chuyển
delivered_orders = datasets["orders"].loc[
    datasets["orders"]["order_status"] == "delivered"
]

missing_shipments = (
    ~delivered_orders["order_id"].isin(
        datasets["shipments"]["order_id"]
    )
).sum()

print("Đơn trước ngày đăng ký:", orders_before_signup)
print("Đơn delivered thiếu vận chuyển:", missing_shipments)

Đơn trước ngày đăng ký: 477453
Đơn delivered thiếu vận chuyển: 524


## 7. Khảo sát và kiểm tra dữ liệu tồn kho (Inventory Data Understanding)

### 7.1. Khảo sát cấu trúc dữ liệu Inventory

In [29]:
inventory = datasets["inventory"].copy()

# Chuyển đổi ngày trên bản sao phục vụ kiểm tra
inventory["snapshot_date"] = pd.to_datetime(
    inventory["snapshot_date"],
    errors="coerce"
)

print("Số dòng, số cột:", inventory.shape)

print("\nDanh sách cột và kiểu dữ liệu:")
display(
    inventory.dtypes
    .astype(str)
    .rename("Data_Type")
    .reset_index()
    .rename(columns={"index": "Column"})
)

print("\n5 dòng dữ liệu mẫu:")
display(inventory.head())

Số dòng, số cột: (60247, 17)

Danh sách cột và kiểu dữ liệu:

5 dòng dữ liệu mẫu:


,Column,Data_Type
0,snapshot_date,datetime64[ns]
1,product_id,int64
2,stock_on_hand,int64
3,units_received,int64
4,units_sold,int64
5,stockout_days,int64
6,days_of_supply,float64
7,fill_rate,float64
8,stockout_flag,int64
9,overstock_flag,int64


,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,fill_rate,stockout_flag,overstock_flag,reorder_flag,sell_through_rate,product_name,category,segment,year,month
0,2022-10-31,1,3,1,1,2,90.0,0.9333,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,10
1,2022-11-30,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,11
2,2022-12-31,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,12
3,2016-04-30,3,35,13,11,2,95.5,0.9333,1,1,0,0.2391,DragonWear MA-03,Casual,All-weather,2016,4
4,2016-05-31,3,36,11,10,1,108.0,0.9667,1,1,0,0.2174,DragonWear MA-03,Casual,All-weather,2016,5


### 7.2 Kiểm tra phạm vi thời gian và sản phẩm

Tổng quan thời gian

In [30]:
inventory_period_report = pd.DataFrame({
    "Metric": [
        "Total_Rows",
        "Unique_Products",
        "Unique_Months",
        "Start_Date",
        "End_Date",
        "Invalid_Or_Missing_Dates",
        "Non_Month_End_Dates"
    ],
    "Result": [
        len(inventory),
        inventory["product_id"].nunique(),
        inventory["snapshot_date"].dt.to_period("M").nunique(),
        inventory["snapshot_date"].min(),
        inventory["snapshot_date"].max(),
        int(inventory["snapshot_date"].isna().sum()),
        int((
            inventory["snapshot_date"].notna()
            & ~inventory["snapshot_date"].dt.is_month_end
        ).sum())
    ]
})

display(inventory_period_report)

,Metric,Result
0,Total_Rows,60247
1,Unique_Products,1624
2,Unique_Months,126
3,Start_Date,2012-07-31 00:00:00
4,End_Date,2022-12-31 00:00:00
5,Invalid_Or_Missing_Dates,0
6,Non_Month_End_Dates,0


Số sản phẩm được ghi nhận theo tháng

In [31]:
inventory_monthly_coverage = (
    inventory
    .groupby("snapshot_date", as_index=False)
    .agg(
        Records=("product_id", "size"),
        Unique_Products=("product_id", "nunique")
    )
    .sort_values("snapshot_date")
)

print("Thống kê số sản phẩm được ghi nhận mỗi tháng:")
display(inventory_monthly_coverage["Unique_Products"].describe())

print("\n5 tháng đầu:")
display(inventory_monthly_coverage.head())

print("\n5 tháng cuối:")
display(inventory_monthly_coverage.tail())

Thống kê số sản phẩm được ghi nhận mỗi tháng:

5 tháng đầu:

5 tháng cuối:


count    126.000000
mean     478.150794
std       40.545889
min      376.000000
25%      454.250000
50%      481.500000
75%      506.500000
max      562.000000
Name: Unique_Products, dtype: float64

,snapshot_date,Records,Unique_Products
0,2012-07-31,395,395
1,2012-08-31,423,423
2,2012-09-30,424,424
3,2012-10-31,437,437
4,2012-11-30,411,411


,snapshot_date,Records,Unique_Products
121,2022-08-31,461,461
122,2022-09-30,419,419
123,2022-10-31,409,409
124,2022-11-30,376,376
125,2022-12-31,424,424


Số tháng có dữ liệu của từng sản phẩm

In [32]:
product_month_coverage = (
    inventory
    .groupby("product_id", as_index=False)
    .agg(
        First_Snapshot=("snapshot_date", "min"),
        Last_Snapshot=("snapshot_date", "max"),
        Observed_Months=("snapshot_date", "nunique")
    )
)

display(product_month_coverage["Observed_Months"].describe())

print("\nMột số sản phẩm có ít tháng ghi nhận nhất:")
display(
    product_month_coverage
    .sort_values("Observed_Months")
    .head(10)
)


Một số sản phẩm có ít tháng ghi nhận nhất:


count    1624.000000
mean       37.097906
std        33.184629
min         1.000000
25%        12.000000
50%        27.000000
75%        53.000000
max       126.000000
Name: Observed_Months, dtype: float64

,product_id,First_Snapshot,Last_Snapshot,Observed_Months
1594,2372,2013-07-31,2013-07-31,1
1540,2246,2013-08-31,2013-08-31,1
1541,2247,2013-08-31,2013-08-31,1
8,11,2014-01-31,2014-01-31,1
35,49,2021-09-30,2021-09-30,1
851,1241,2021-02-28,2021-02-28,1
34,48,2021-09-30,2021-09-30,1
86,173,2022-12-31,2022-12-31,1
114,210,2022-12-31,2022-12-31,1
1462,2127,2012-10-31,2012-10-31,1


### 7.3 Kiểm tra giá trị số và các chỉ số tồn kho

Thống kê mô tả

In [33]:
inventory_numeric_cols = [
    "stock_on_hand",
    "units_received",
    "units_sold",
    "stockout_days",
    "days_of_supply",
    "fill_rate",
    "sell_through_rate"
]

inventory_numeric_summary = (
    inventory[inventory_numeric_cols]
    .describe()
    .T
    [["count", "mean", "50%", "min", "max"]]
    .rename(columns={
        "count": "Count",
        "mean": "Mean",
        "50%": "Median",
        "min": "Min",
        "max": "Max"
    })
)

display(inventory_numeric_summary.round(4))

,Count,Mean,Median,Min,Max
stock_on_hand,60247.0,189.2985,62.0000,3.0000,2673.0000
units_received,60247.0,18.0468,6.0000,1.0000,817.0000
units_sold,60247.0,15.4178,6.0000,1.0000,670.0000
stockout_days,60247.0,1.1606,1.0000,0.0000,28.0000
days_of_supply,60247.0,912.6776,240.0000,5.2000,68100.0000
fill_rate,60247.0,0.9613,0.9667,0.0667,1.0000
sell_through_rate,60247.0,0.1523,0.1111,0.0004,0.8531


Kiểm tra phạm vi giá trị

In [34]:
inventory_numeric_checks = {
    "Negative_Stock_On_Hand":
        inventory["stock_on_hand"] < 0,

    "Negative_Units_Received":
        inventory["units_received"] < 0,

    "Negative_Units_Sold":
        inventory["units_sold"] < 0,

    "Negative_Stockout_Days":
        inventory["stockout_days"] < 0,

    "Negative_Days_Of_Supply":
        inventory["days_of_supply"] < 0,

    "Fill_Rate_Outside_0_1":
        ~inventory["fill_rate"].between(0, 1),

    "Sell_Through_Rate_Outside_0_1":
        ~inventory["sell_through_rate"].between(0, 1),

    "Stockout_Days_Above_31":
        inventory["stockout_days"] > 31
}

inventory_numeric_check_report = pd.DataFrame([
    {
        "Check": name,
        "Count": int(mask.sum())
    }
    for name, mask in inventory_numeric_checks.items()
])

display(inventory_numeric_check_report)

,Check,Count
0,Negative_Stock_On_Hand,0
1,Negative_Units_Received,0
2,Negative_Units_Sold,0
3,Negative_Stockout_Days,0
4,Negative_Days_Of_Supply,0
5,Fill_Rate_Outside_0_1,0
6,Sell_Through_Rate_Outside_0_1,0
7,Stockout_Days_Above_31,0


Xem các giá trị days_of_supply lớn nhất

In [35]:
high_dos_records = (
    inventory[
        [
            "snapshot_date",
            "product_id",
            "product_name",
            "stock_on_hand",
            "units_sold",
            "days_of_supply"
        ]
    ]
    .sort_values("days_of_supply", ascending=False)
    .head(20)
)

display(high_dos_records)

,snapshot_date,product_id,product_name,stock_on_hand,units_sold,days_of_supply
10707,2020-12-31,604,SaigonFlex UC-69,2270,1,68100.0
6841,2022-05-31,487,SaigonFlex UM-92,2014,1,60420.0
6837,2022-01-31,487,SaigonFlex UM-92,2011,1,60330.0
6836,2021-12-31,487,SaigonFlex UM-92,2011,1,60330.0
5963,2022-11-30,438,SaigonFlex UM-43,1744,1,52320.0
5961,2022-06-30,438,SaigonFlex UM-43,1744,1,52320.0
5960,2022-05-31,438,SaigonFlex UM-43,1744,1,52320.0
5957,2021-10-31,438,SaigonFlex UM-43,1744,1,52320.0
5958,2022-02-28,438,SaigonFlex UM-43,1744,1,52320.0
5959,2022-04-30,438,SaigonFlex UM-43,1744,1,52320.0


### 7.4 Kiểm tra các cờ cảnh báo tồn kho

In [36]:
inventory_flags = [
    "stockout_flag",
    "overstock_flag",
    "reorder_flag"
]

flag_records = []

for col in inventory_flags:
    counts = inventory[col].value_counts(dropna=False)

    for value, count in counts.items():
        flag_records.append({
            "Flag": col,
            "Value": value,
            "Count": int(count),
            "Percentage": round(
                count / len(inventory) * 100, 2
            )
        })

inventory_flag_report = pd.DataFrame(flag_records)

display(inventory_flag_report)

# Kiểm tra hai cảnh báo cùng xuất hiện
both_flags = ( (inventory["stockout_flag"] == 1) & (inventory["overstock_flag"] == 1))

print("Số bản ghi đồng thời stockout và overstock:", int(both_flags.sum()))

print( "Tỷ lệ trên tổng số bản ghi:", round(both_flags.mean() * 100, 2), "%")

Số bản ghi đồng thời stockout và overstock: 30495
Tỷ lệ trên tổng số bản ghi: 50.62 %


,Flag,Value,Count,Percentage
0,stockout_flag,1,40571,67.34
1,stockout_flag,0,19676,32.66
2,overstock_flag,1,45942,76.26
3,overstock_flag,0,14305,23.74
4,reorder_flag,0,60247,100.00


### 7.5. Kiểm tra tính nhất quán của cảnh báo hết hàng

$$\text{stockout\_flag}=1 \iff \text{stockout\_days}>0$$


In [37]:
expected_stockout_flag = (
    inventory["stockout_days"] > 0
).astype(int)

stockout_mismatch_mask = (
    inventory["stockout_flag"] != expected_stockout_flag
)

stockout_consistency_report = pd.DataFrame({
    "Metric": [
        "Rows_With_Stockout_Days_Above_Zero",
        "Rows_With_Stockout_Flag_One",
        "Inconsistent_Rows"
    ],
    "Count": [
        int((inventory["stockout_days"] > 0).sum()),
        int((inventory["stockout_flag"] == 1).sum()),
        int(stockout_mismatch_mask.sum())
    ]
})

display(stockout_consistency_report)

,Metric,Count
0,Rows_With_Stockout_Days_Above_Zero,40571
1,Rows_With_Stockout_Flag_One,40571
2,Inconsistent_Rows,0


### 7.6 Đối chiếu tồn kho giữa các tháng (Inventory Reconciliation)

Kiểm tra tính nhất quán giữa tồn kho cuối tháng, lượng nhập và lượng bán của từng sản phẩm.

Trong bộ dữ liệu này, chúng ta có thể kiểm tra:

$$\text{Expected Stock}_t = \text{Stock On Hand}_{t-1} + \text{Units Received}_t - \text{Units Sold}_t$$

Tính sai lệch:

$$\text{Difference}_t = \text{Actual Stock}_t - \text{Expected Stock}_t$$

Chuẩn bị dữ liệu đối chiếu


In [38]:
# Tạo bản sao, không thay đổi dữ liệu đã đọc
inventory_reconciliation = (
    inventory
    .sort_values(["product_id", "snapshot_date"])
    .copy()
)

# Lấy thông tin của bản ghi trước đó trong cùng sản phẩm
grouped_inventory = inventory_reconciliation.groupby("product_id")

inventory_reconciliation["previous_date"] = (
    grouped_inventory["snapshot_date"].shift(1)
)

inventory_reconciliation["previous_stock"] = (
    grouped_inventory["stock_on_hand"].shift(1)
)

# Xác định tháng hiện tại và tháng trước
current_month = (
    inventory_reconciliation["snapshot_date"]
    .dt.to_period("M")
)

previous_month = (
    inventory_reconciliation["previous_date"]
    .dt.to_period("M")
)

# Chỉ chọn những cặp tháng thực sự liên tiếp
consecutive_month_mask = (
    previous_month.notna()
    & (current_month == previous_month + 1)
)

# Tạo bảng chỉ chứa những cặp tháng hợp lệ
inventory_consecutive = (
    inventory_reconciliation.loc[
        consecutive_month_mask
    ].copy()
)

print(
    "Số cặp tháng liên tiếp hợp lệ:",
    len(inventory_consecutive)
)

Số cặp tháng liên tiếp hợp lệ: 51524


Tính phương trình tồn kho

In [39]:
# Tồn kho cuối kỳ dự kiến
inventory_consecutive["expected_stock"] = (
    inventory_consecutive["previous_stock"]
    + inventory_consecutive["units_received"]
    - inventory_consecutive["units_sold"]
)

# Sai lệch giữa tồn thực tế và tồn dự kiến
inventory_consecutive["stock_difference"] = (
    inventory_consecutive["stock_on_hand"]
    - inventory_consecutive["expected_stock"]
)

# Đánh dấu các cặp cân bằng
inventory_consecutive["is_balanced"] = (
    inventory_consecutive["stock_difference"] == 0
)

# Tổng hợp kết quả
inventory_reconciliation_report = pd.DataFrame({
    "Metric": [
        "Consecutive_Month_Pairs",
        "Balanced_Pairs",
        "Unbalanced_Pairs",
        "Balance_Rate_Percent",
        "Maximum_Absolute_Difference"
    ],
    "Result": [
        len(inventory_consecutive),
        int(inventory_consecutive["is_balanced"].sum()),
        int((~inventory_consecutive["is_balanced"]).sum()),
        round(
            inventory_consecutive["is_balanced"].mean() * 100,
            2
        ),
        inventory_consecutive["stock_difference"].abs().max()
    ]
})

display(inventory_reconciliation_report)

,Metric,Result
0,Consecutive_Month_Pairs,51524.0
1,Balanced_Pairs,51524.0
2,Unbalanced_Pairs,0.0
3,Balance_Rate_Percent,100.0
4,Maximum_Absolute_Difference,0.0


### 7.7 Đối chiếu số lượng bán giữa dữ liệu giao dịch và tồn kho

#### 7.7.1 Đối chiếu tổng số lượng sản phẩm

Tổng units_sold được ghi nhận trong bảng inventory có tương đồng với lượng đặt mua và lượng giao thành công tính từ dữ liệu giao dịch không?

In [40]:

# Tổng số lượng sản phẩm được đặt mua
total_ordered = order_items["quantity"].sum()

# Lấy trạng thái đơn hàng cho từng dòng sản phẩm
order_items_status = order_items.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

# Tổng số lượng thuộc các đơn delivered
total_delivered = order_items_status.loc[
    order_items_status["order_status"] == "delivered",
    "quantity"
].sum()

# Tổng lượng bán ghi nhận trong Inventory
total_inventory_sold = inventory["units_sold"].sum()

# Bảng đối chiếu
quantity_comparison = pd.DataFrame({
    "Metric": [
        "Ordered Quantity",
        "Delivered Quantity",
        "Inventory Units Sold"
    ],
    "Total_Units": [
        total_ordered,
        total_delivered,
        total_inventory_sold
    ]
})

display(quantity_comparison)


,Metric,Total_Units
0,Ordered Quantity,3213143
1,Delivered Quantity,2565930
2,Inventory Units Sold,928874


#### 7.7.2 Đối chiếu số lượng bán theo sản phẩm–tháng

In [41]:

# Lấy các dòng sản phẩm thuộc đơn delivered
delivered_items_check = order_items_status[
    order_items_status["order_status"] == "delivered"
].copy()

# Ghép ngày đặt hàng vào dữ liệu giao dịch
delivered_items_check = delivered_items_check.merge(
    orders[["order_id", "order_date"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

# Xác định tháng đặt hàng
delivered_items_check["year_month"] = (
    pd.to_datetime(delivered_items_check["order_date"])
    .dt.to_period("M")
)

# Tổng Delivered Quantity theo sản phẩm-tháng
monthly_delivered = (
    delivered_items_check
    .groupby(["product_id", "year_month"])["quantity"]
    .sum()
    .reset_index(name="delivered_quantity")
)

# Tổng Units Sold theo sản phẩm-tháng
inventory_check = inventory[
    ["product_id", "snapshot_date", "units_sold"]
].copy()

inventory_check["year_month"] = (
    pd.to_datetime(inventory_check["snapshot_date"])
    .dt.to_period("M")
)

monthly_inventory = (
    inventory_check
    .groupby(["product_id", "year_month"])["units_sold"]
    .sum()
    .reset_index(name="inventory_units_sold")
)

# Ghép để xác định phạm vi xuất hiện ở hai nguồn
quantity_detail = monthly_delivered.merge(
    monthly_inventory,
    on=["product_id", "year_month"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

# Thống kê phạm vi đối chiếu
coverage_summary = (
    quantity_detail["_merge"]
    .value_counts()
    .reindex(["both", "left_only", "right_only"], fill_value=0)
    .rename(index={
        "both": "Có ở cả hai nguồn",
        "left_only": "Chỉ có ở giao dịch",
        "right_only": "Chỉ có ở inventory"
    })
    .reset_index()
)

coverage_summary.columns = [
    "Nguồn dữ liệu",
    "Số sản phẩm-tháng"
]

display(coverage_summary)


,Nguồn dữ liệu,Số sản phẩm-tháng
0,Có ở cả hai nguồn,53965
1,Chỉ có ở giao dịch,570
2,Chỉ có ở inventory,6282


Kiểm tra mức độ khớp về số lượng

In [42]:

# Chỉ xét các sản phẩm-tháng xuất hiện ở cả hai nguồn
matched_quantity = quantity_detail[
    quantity_detail["_merge"] == "both"
].copy()

# Tính chênh lệch số lượng
matched_quantity["quantity_difference"] = (
    matched_quantity["delivered_quantity"]
    - matched_quantity["inventory_units_sold"]
)

# Kiểm tra số cặp có lượng bán bằng nhau
exact_matches = (
    matched_quantity["quantity_difference"] == 0
).sum()

total_matched = len(matched_quantity)

print(f"Total Matched Product-Months: {total_matched:,}")
print(f"Exact Matches: {exact_matches:,}")
print(f"Different Quantities: {total_matched - exact_matches:,}")
print(f"Exact Match Rate: {exact_matches / total_matched * 100:.2f}%")

# Tổng hợp số lượng trên cùng phạm vi
print("\nQuantity Comparison (Matched Records)")
print(
    "Delivered Quantity:",
    f"{matched_quantity['delivered_quantity'].sum():,}"
)
print(
    "Inventory Units Sold:",
    f"{matched_quantity['inventory_units_sold'].sum():,}"
)
print(
    "Total Difference:",
    f"{matched_quantity['quantity_difference'].sum():,}"
)

# Thống kê mức độ chênh lệch tuyệt đối
display(
    matched_quantity["quantity_difference"]
    .abs()
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95])
)


Total Matched Product-Months: 53,965
Exact Matches: 1,922
Different Quantities: 52,043
Exact Match Rate: 3.56%

Quantity Comparison (Matched Records)
Delivered Quantity: 2,563,228.0
Inventory Units Sold: 920,263.0
Total Difference: 1,642,965.0


count    53965.000000
mean        30.562902
std         53.107623
min          0.000000
50%         12.000000
75%         33.000000
90%         78.000000
95%        121.000000
max       1362.000000
Name: quantity_difference, dtype: float64

In [43]:

# Phân loại chiều hướng chênh lệch
greater_count = (
    matched_quantity["quantity_difference"] > 0
).sum()

lower_count = (
    matched_quantity["quantity_difference"] < 0
).sum()

equal_count = (
    matched_quantity["quantity_difference"] == 0
).sum()

difference_direction = pd.DataFrame({
    "Comparison": [
        "Delivered > Inventory",
        "Delivered < Inventory",
        "Delivered = Inventory"
    ],
    "Product_Months": [
        greater_count,
        lower_count,
        equal_count
    ]
})

difference_direction["Percentage"] = (
    difference_direction["Product_Months"]
    / len(matched_quantity) * 100
).round(2)

display(difference_direction)


,Comparison,Product_Months,Percentage
0,Delivered > Inventory,50356,93.31
1,Delivered < Inventory,1687,3.13
2,Delivered = Inventory,1922,3.56


**Nhận xét:** Kết quả đối chiếu cho thấy số lượng bán ghi nhận trong dữ liệu giao dịch và dữ liệu tồn kho có sự khác biệt đáng kể. Trên tổng thể, Delivered Quantity đạt 2.565.930 Units, trong khi Inventory Units Sold chỉ đạt 928.874 Units. Ở cấp sản phẩm–tháng, có 53.965 cặp xuất hiện đồng thời ở cả hai nguồn, nhưng chỉ 1.922 cặp (3,56%) có số lượng khớp chính xác. Đáng chú ý, 93,31% số cặp ghi nhận Delivered Quantity lớn hơn Inventory Units Sold, cho thấy sai lệch có xu hướng chủ yếu theo một chiều.

Kết quả cho thấy hai chỉ số chưa thể được xem là tương đương hoặc sử dụng thay thế trực tiếp cho nhau. Tuy nhiên, do Delivered Quantity được xác định từ trạng thái đơn hàng và ngày đặt hàng, trong khi Inventory Units Sold được ghi nhận theo dữ liệu tồn kho hàng tháng, sự khác biệt chưa đủ để khẳng định có lỗi dữ liệu. Cần làm rõ quy tắc ghi nhận và phạm vi nghiệp vụ trước khi đưa ra quyết định xử lý.

Trong các phân tích tiếp theo, Ordered Quantity, Delivered Quantity và Inventory Units Sold sẽ được giữ riêng biệt theo đúng nguồn dữ liệu. Không tự động điều chỉnh hoặc đồng nhất các giá trị khi chưa có căn cứ nghiệp vụ.